# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the "A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya" using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{getattr(metadata, 'name', 'Dataset')}: {getattr(metadata, 'description', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs for structured navigation of the dataset. All references use Croissant `@id` identifiers.

In [ ]:
# List all available record sets (@id and name)
print("Available Record Sets:")
record_sets = list(dataset.record_sets)
for rs in record_sets:
    print(f"  - @id: {rs['@id']}, name: {rs.get('name','')}")

# Let's choose the main survey record set for illustration (commonly named, or pick the first)
if len(record_sets) > 0:
    main_record_set_id = record_sets[0]['@id']
    main_record_set = dataset.get_record_set(main_record_set_id)
    print(f"\nFields in record set {main_record_set_id}:")
    for field in main_record_set['fields']:
        print(f"  - @id: {field['@id']}, name: {field.get('name','')}, dataType: {field.get('dataType','')}")
else:
    print('No record sets found in this dataset.')

## 3. Data Extraction
Load all record sets into pandas DataFrames for analysis. We'll use the Croissant record set `@id` values identified in the previous step.

In [ ]:
# Extract data from all record sets into DataFrames
dataframes = {}

for rs in record_sets:
    rs_id = rs['@id']
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"Loaded {df.shape[0]} records and {df.shape[1]} columns for record set {rs_id}.")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

# Display columns for the main record set
print(f"\nAvailable columns in main record set ({main_record_set_id}):")
df = dataframes.get(main_record_set_id)
if df is not None:
    print(df.columns.tolist())
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Let's process, filter, and analyze survey data. For this example, we'll use fields by their Croissant `@id`. Common analyses include filtering, normalization, and group statistics.

In [ ]:
# Example using the GAD-7 total score (typical numeric field, update the name if different per schema)
# We use the field @id—update with your specific numeric @id if different
numeric_field_id = None
for field in main_record_set['fields']:
    # Look for a GAD-7 total score, PHQ-9, or PSQ field, adjust as needed
    if 'gad7' in field['@id'].lower() or 'GAD' in field.get('name',''):
        numeric_field_id = field['@id']
        break
if numeric_field_id is None:
    # Fallback to any float/integer field (demonstration only)
    for field in main_record_set['fields']:
        if field.get('dataType','').lower() in ['integer', 'float', 'number']:
            numeric_field_id = field['@id']
            break

df = dataframes.get(main_record_set_id)
if df is not None and numeric_field_id in df.columns:
    # Remove missing values
    df_numeric = df[df[numeric_field_id].notna()].copy()
    threshold = 10
    filtered_df = df_numeric[df_numeric[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Try grouping by gender field (example)
    group_field_id = None
    for field in main_record_set['fields']:
        if 'gender' in field['@id'].lower() or 'gender' in field.get('name','').lower():
            group_field_id = field['@id']
            break

    if group_field_id and group_field_id in df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
        print(f"\nMean {numeric_field_id} by {group_field_id}:")
        print(grouped_df)
else:
    print("No suitable numeric field found in main record set.")

## 5. Visualization
Visualize the distribution of a chosen numeric field (e.g., GAD-7 score) and compare by demographic (e.g., gender).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load, explore, and perform basic processing of the Kilifi County mental health survey dataset using the `mlcroissant` library. We reviewed the dataset's metadata, dynamically listed record sets and their fields, loaded the main record set into a DataFrame, and visualized a key mental health score field. For more advanced analysis, further domain-specific field selection and statistical tests can be performed using the same methods.